# Fig. 1h — Spatial Broad Cell-Type Plots (Whole Image, Dark Background)

**Purpose:** Generate whole-tissue spatial plots of predicted broad cell
types (from Tangram deconvolution) for Figure 1h, styled with a dark/black
background and a "neon" color palette matching the original manuscript
figure.

For each sample, this produces three PDF plots:
- All predicted cell types, whole tissue
- All predicted cell types, whole tissue, **excluding hepatocytes** (to make
  rarer cell types easier to see against the abundant hepatocyte signal)
- The same "no hepatocytes" view with the underlying H&E image alpha set to 0
  (i.e. cell-type markers only, no tissue image)

**Inputs:** filtered, Tangram-annotated `.h5ad` files per sample
(`adata_0.5_tangram_filt.h5ad`), one per `samples` entry.

**Outputs:** one subdirectory of PDFs per sample under `odir`.

**Cleanup notes (this pass):** removed a large block of commented-out,
never-used code (a per-cell-type plotting loop, expression-heatmap and
n_counts-distribution calls referencing functions not defined in this
notebook), fixed a syntax-breaking stray indent on the marker gene list, and
removed a duplicate list entry. No plotting logic that produces the saved
PDFs was changed.

In [3]:
import os

import matplotlib.pyplot as plt
import scanpy as sc


## Sample-processing driver

In [4]:
def process_sample(samples, color_map, gene_markers, idir, filt_h5ad_dir, odir):
    """
    Generate spatial cell-type plots for a list of samples.

    Parameters
    ----------
    samples : list[str]
        Sample names to process.
    color_map : dict
        Mapping of cell type -> hex color for plotting.
    gene_markers : list[str]
        Marker genes (currently unused here; kept for interface
        compatibility with expression-heatmap plotting used elsewhere).
    idir : str
        Input directory containing spatial/Tangram outputs (currently unused
        directly in this function; kept for interface compatibility).
    filt_h5ad_dir : str
        Directory containing per-sample filtered, Tangram-annotated
        `.h5ad` files (`<filt_h5ad_dir>/<sample>/adata_0.5_tangram_filt.h5ad`).
    odir : str
        Output directory; a per-sample subdirectory is created under it.
    """
    os.makedirs(odir, exist_ok=True)
    for sample_nm in samples:
        print(f"Processing {sample_nm}...")

        filt_h5ad_sample_dir = os.path.join(filt_h5ad_dir, sample_nm)
        ad_sp_file = os.path.join(filt_h5ad_sample_dir, "adata_0.5_tangram_filt.h5ad")
        patdir = os.path.join(odir, sample_nm)
        os.makedirs(patdir, exist_ok=True)

        adata_pred_filt = sc.read_h5ad(ad_sp_file)

        # Plot spatial data by cell type
        plot_spatial_by_celltype(adata_pred_filt, outdir=patdir, sample_nm=sample_nm, palette_dict=color_map)


## Core plotting function

In [5]:
def plot_spatial_by_celltype(adata, outdir, sample_nm, palette_dict):
    """
    Generate whole-tissue spatial plots of predicted cell types, saved as
    black-background PDFs.

    Produces three plots per sample:
    1. All cell types, whole tissue.
    2. All cell types except hepatocytes, whole tissue (hepatocytes excluded
       to make rarer populations easier to see).
    3. Same as (2), but with the H&E image alpha set to 0 (markers only).

    Parameters
    ----------
    adata : AnnData
        Spatial data with a `pred_cell_types` column in `.obs`.
    outdir : str
        Directory to save the PDFs into.
    sample_nm : str
        Sample name, used in output filenames.
    palette_dict : dict
        Mapping of cell type -> hex color.
    """
    sc.set_figure_params(dpi_save=500)

    unique_subclusters = adata.obs['pred_cell_types'].unique()

    # --- Plot 1: all cell types, whole tissue ---
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('black')
    ax.set_facecolor('black')

    sc.pl.spatial(
        adata,
        alpha_img=0,
        palette=[palette_dict.get(celltype, '#000000') for celltype in sorted(unique_subclusters)],
        color='pred_cell_types',
        ax=ax,
        size=2,
        show=False,
    )

    legend = ax.get_legend()
    if legend is not None:
        for text in legend.get_texts():
            text.set_color('white')

    plt.savefig(f"{outdir}/{sample_nm}_anno_broad_whole_tissue.pdf", bbox_inches='tight')
    plt.close(fig)

    # --- Plot 2: all cell types except hepatocytes, whole tissue ---
    adata_noheps = adata[~adata.obs['pred_cell_types'].isin(['Hepatocytes'])]
    unique_subclusters_noheps = adata.obs['pred_cell_types'].unique()

    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('black')
    ax.set_facecolor('black')

    sc.pl.spatial(
        adata_noheps,
        alpha_img=0,
        palette=[palette_dict.get(celltype, '#000000') for celltype in sorted(unique_subclusters_noheps)],
        color='pred_cell_types',
        ax=ax,
        size=2,
        show=False,
    )

    legend = ax.get_legend()
    if legend is not None:
        for text in legend.get_texts():
            text.set_color('white')

    plt.savefig(f"{outdir}/{sample_nm}_anno_broad_whole_tissue_noheps.pdf", bbox_inches='tight')
    plt.close(fig)

    # --- Plot 3: same as (2), but markers only (H&E image alpha = 0) ---
    print("Plotting H&E")
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('black')
    ax.set_facecolor('black')

    sc.pl.spatial(
        adata_noheps,
        palette=[palette_dict.get(celltype, '#000000') for celltype in sorted(unique_subclusters_noheps)],
        color='pred_cell_types',
        ax=ax,
        size=2,
        show=False,
        alpha=0,
    )
    plt.savefig(f"{outdir}/{sample_nm}_anno_broad_whole_tissue_he.pdf", bbox_inches='tight')
    plt.close(fig)


## Color palette and marker genes

Neon color palette matching the original manuscript figure (broad cell-type coloring; see comment below for the alternate,
non-neon hex codes also used in the paper).

In [6]:
# from paper very original
neon_rainbow_colors_broad = {
        'Hepatocytes': '#FF007F',    # Neon pink #maroon is '#800000'
        'Myeloid': '#0000FF',           # Neon blue
        'T_NK': '#FFFA00',              # Neon yellow
        'Cholangiocyte': '#00FF00',  # Neon green
        'HSC': '#00FFFF',            # Neon cyan
        'Mast': '#FF1900',        # Neon red
        'Endothelial': '#8A2BE2',    # Neon purple
        'Schwann': '#FF1493',        # Neon deep pink
        'NK': '#FFFDBB',             # lighter yellow
        'B': '#FF8800'               # Neon hot pink
    }

Alternate (non-neon) hex codes used elsewhere in the paper for the same
broad cell types, kept here for reference:
- Hepatocytes — `#ec2d4b`
- Cholangiocyte — `#5baf5c`
- Endothelial — `#633d99`
- HSC — `#75c2e0`
- Myeloid — `#474d9c`
- T/NK — `#f6ed24`
- B — `#f78a1f`

In [8]:
brin_markers = [
    "cd19", "ms4a1",
    "krt19", "fxyd2", "spp1",
    "epcam", "sox9", "anxa4", "sry", "krt1",
    "pecam1", "sele", "flt4",
    "lyve1", "mcam", "cd34", "ptprc", "stab2", "ptprb",  # endothelial
    "col1a1", "fap", "adamts13",
    "ngfr", "cygb", "hgf", "rbp1",  # stellate cells
    "msln", "thy1",  # fibroblasts
    "grem1", "aspn", "calca", "eln",  # Gremlin1, Asporin, calcitonin a, Elastin (fibroblasts)
    "cyp2e1", "hnf4a", "crp", "alb",
    "serpina1", "ttr",  # hepatocytes
    "c1qa", "cd163", "timd4",
    "cd68", "ms4a7",  # myeloid
    "cd69", "trbc2", "cd3d",
]


## Configuration and run

In [9]:
idir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/tangram/deep_seq/'
filt_h5ad_dir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/misc/tangram_scores_dist2/'
odir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/tangram/misc/qc_plots_whole_image_broad_ct_darkbg/'


In [10]:
samples = ['HL160029_filt', 'HL230324_filt', 'HL20221019_broad_no_mast_filt', 'HL170058_filt']


In [11]:
process_sample(
    samples,
    neon_rainbow_colors_broad,
    brin_markers,
    idir,
    filt_h5ad_dir,
    odir,
)


Processing HL160029_filt...


/tscc/nfs/home/cmiciano/miniconda3/envs/spatial/lib/python3.11/site-packages/scanpy/plotting/_utils.py:465: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns[value_to_plot + "_colors"] = colors_list


Plotting H&E
Processing HL230324_filt...


/tscc/nfs/home/cmiciano/miniconda3/envs/spatial/lib/python3.11/site-packages/scanpy/plotting/_utils.py:465: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns[value_to_plot + "_colors"] = colors_list


Plotting H&E
Processing HL20221019_broad_no_mast_filt...


/tscc/nfs/home/cmiciano/miniconda3/envs/spatial/lib/python3.11/site-packages/scanpy/plotting/_utils.py:465: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns[value_to_plot + "_colors"] = colors_list


Plotting H&E
Processing HL170058_filt...


/tscc/nfs/home/cmiciano/miniconda3/envs/spatial/lib/python3.11/site-packages/scanpy/plotting/_utils.py:465: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns[value_to_plot + "_colors"] = colors_list


Plotting H&E
